CELL 1 — Imports

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

CELL 2 — Project Configuration

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/expo"

ARTIFACT_DIR = f"{PROJECT_ROOT}/artifacts"
MODEL_DIR = f"{PROJECT_ROOT}/models"

os.makedirs(MODEL_DIR, exist_ok=True)

CELL 3 — Load Artifacts

In [ ]:
hotel_matrix = pd.read_parquet(
    f"{ARTIFACT_DIR}/hotel_matrix.parquet"
)

consistency_matrix = pd.read_parquet(
    f"{ARTIFACT_DIR}/consistency_matrix.parquet"
)

mention_matrix = pd.read_parquet(
    f"{ARTIFACT_DIR}/mention_matrix.parquet"
)

hotel_trends = pd.read_parquet(
    f"{ARTIFACT_DIR}/hotel_trends.parquet"
)

traveler_feature_store = pd.read_parquet(
    f"{ARTIFACT_DIR}/traveler_feature_store.parquet"
)

hotel_contradictions = pd.read_parquet(
    f"{ARTIFACT_DIR}/hotel_contradictions.parquet"
)

hotel_confidence = pd.read_parquet(
    f"{ARTIFACT_DIR}/hotel_confidence_scores.parquet"
)

profiles = pd.read_parquet(
    f"{ARTIFACT_DIR}/profiles_clean.parquet"
)

reviews = pd.read_parquet(
    f"{ARTIFACT_DIR}/reviews_clean.parquet"
)

CELL 4 — Load Embedding Model

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

CELL 5 — Aspect Embeddings

In [ ]:
with open(
    f"{ARTIFACT_DIR}/aspect_descriptions.json"
) as f:
    aspect_descriptions = json.load(f)

ASPECTS = list(
    aspect_descriptions.keys()
)

aspect_embeddings = model.encode(
    list(
        aspect_descriptions.values()
    ),
    normalize_embeddings=True
)

CELL 6 — Convert Profiles Into Aspect Preferences

In [ ]:
profile_embeddings = model.encode(
    profiles["description"].tolist(),
    normalize_embeddings=True
)

similarity_matrix = cosine_similarity(
    profile_embeddings,
    aspect_embeddings
)

profile_vectors = pd.DataFrame(
    similarity_matrix,
    columns=ASPECTS
)

profile_vectors.insert(
    0,
    "profile_id",
    profiles["profile_id"]
)

profile_vectors.head()

,profile_id,safety,location,culture,business,family,wellness,service,cleanliness,value,accessibility,nightlife,beach
0,P01,0.161367,0.430223,0.360864,0.114095,0.060044,0.129214,0.236044,0.089922,0.163050,0.201683,0.221065,0.108414
1,P02,0.246582,0.300098,0.097971,0.454948,0.158050,0.115764,0.265696,0.113537,0.264391,0.187699,0.139541,0.132122
2,P03,0.019596,0.330215,0.175476,0.110706,0.516090,0.045700,0.215400,0.053767,0.164703,0.150167,0.128679,0.110131
3,P04,0.091019,0.285386,0.254690,0.191016,0.134242,0.380247,0.339989,0.140474,0.206005,0.057064,0.228128,0.069766
4,P05,0.165998,0.398459,0.419503,0.130856,0.101346,0.153479,0.253422,0.015325,0.437977,0.067410,0.297711,0.116740


CELL 7 — Normalize Preferences

In [ ]:
scaler = MinMaxScaler()

profile_vectors[
    ASPECTS
] = scaler.fit_transform(
    profile_vectors[
        ASPECTS
    ]
)

profile_vectors.head()

,profile_id,safety,location,culture,business,family,wellness,service,cleanliness,value,accessibility,nightlife,beach
0,P01,0.617601,0.774691,0.498354,0.044221,0.094415,0.251057,0.359167,0.376931,0.250539,0.333504,0.401347,0.283661
1,P02,0.917821,0.371810,0.060393,0.907844,0.248154,0.220870,0.454044,0.439942,0.526799,0.313545,0.249454,0.328233
2,P03,0.118130,0.465055,0.189511,0.035635,0.809799,0.063618,0.293109,0.280459,0.255046,0.259976,0.229217,0.286888
3,P04,0.369761,0.326262,0.321475,0.239117,0.210807,0.814477,0.691761,0.511818,0.367636,0.127091,0.414505,0.211001
4,P05,0.633918,0.676346,0.596043,0.086687,0.159204,0.305519,0.414770,0.177883,1.000000,0.141858,0.544150,0.299314


CELL 8 — Mention Confidence Matrix

In [ ]:
mention_weight_matrix = np.log1p(
    mention_matrix
)

mention_weight_matrix = (
    mention_weight_matrix
    /
    mention_weight_matrix.max()
)

CELL 9 — Trend Matrix

In [ ]:
trend_matrix = (
    hotel_trends
    .pivot(
        index="hotel_id",
        columns="aspect",
        values="trend_score"
    )
)

trend_matrix = trend_matrix.fillna(0)

CELL 10 — Ranking Function

In [ ]:
def rank_hotels(profile_id):

    profile = (
        profile_vectors[
            profile_vectors["profile_id"] == profile_id
        ]
        .iloc[0]
    )

    profile_weights = profile[ASPECTS].values

    scores = []

    for hotel_id in hotel_matrix.index:

        # --------------------------------------------------
        # Hotel feature vectors
        # --------------------------------------------------

        hotel_vector = (
            hotel_matrix
            .loc[hotel_id]
            .reindex(ASPECTS)
            .values
        )

        confidence_vector = (
            mention_weight_matrix
            .loc[hotel_id]
            .reindex(ASPECTS)
            .values
        )

        consistency_vector = (
            consistency_matrix
            .loc[hotel_id]
            .reindex(ASPECTS)
            .values
        )

        trend_vector = (
            trend_matrix
            .reindex(columns=ASPECTS)
            .loc[hotel_id]
            .fillna(0)
            .values
        )

        # --------------------------------------------------
        # Core profile matching
        # --------------------------------------------------

        profile_match = np.sum(
            profile_weights * hotel_vector
        )

        # --------------------------------------------------
        # Evidence confidence bonus
        # --------------------------------------------------

        confidence_bonus = np.sum(
            profile_weights * confidence_vector
        )

        # --------------------------------------------------
        # Consistency bonus
        # --------------------------------------------------

        consistency_bonus = np.sum(
            profile_weights * consistency_vector
        )

        # --------------------------------------------------
        # Trend bonus / penalty
        # --------------------------------------------------

        trend_adjustment = np.sum(
            profile_weights * trend_vector
        )

        # --------------------------------------------------
        # Contradiction penalty
        # --------------------------------------------------

        contradiction_penalty = 0

        hotel_cons = hotel_contradictions[
            hotel_contradictions["hotel_id"] == hotel_id
        ]

        for _, row in hotel_cons.iterrows():

            aspect = row["aspect"]

            if (
                aspect in ASPECTS
                and profile[aspect] > 0.70
            ):
                contradiction_penalty += (
                    0.15 * profile[aspect]
                )

        # --------------------------------------------------
        # Hotel-level confidence adjustment
        # --------------------------------------------------

        hotel_level_confidence = float(
            hotel_confidence[
                hotel_confidence["hotel_id"] == hotel_id
            ]["confidence_score"].iloc[0]
        )

        hotel_level_confidence = (
            hotel_level_confidence
            -
            hotel_confidence["confidence_score"].min()
        ) / (
            hotel_confidence["confidence_score"].max()
            -
            hotel_confidence["confidence_score"].min()
        )

        # --------------------------------------------------
        # Final ranking score
        # --------------------------------------------------

        final_score = (
            0.55 * profile_match
            + 0.10 * confidence_bonus
            + 0.10 * consistency_bonus
            + 0.15 * trend_adjustment
            + 0.10 * hotel_level_confidence
            - contradiction_penalty
        )

        scores.append(
            (
                hotel_id,
                final_score
            )
        )

    # --------------------------------------------------
    # Normalize to recommendation score
    # --------------------------------------------------

    scores = pd.DataFrame(
        scores,
        columns=[
            "hotel_id",
            "raw_score"
        ]
    )

    scores["score"] = (
        scores["raw_score"]
        -
        scores["raw_score"].min()
    )

    denominator = scores["score"].max()

    if denominator > 0:
        scores["score"] = (
            scores["score"]
            / denominator
        )

    # Avoid perfect certainty scores
    scores["score"] = (
        scores["score"] * 95
    ) + 5

    scores = scores.sort_values(
        "score",
        ascending=False
    )

    return list(
        zip(
            scores["hotel_id"],
            scores["score"]
        )
    )[:5]

CELL 11 — Generate Explanations

In [ ]:
def generate_explanation(
    profile_id,
    hotel_id
):

    REASON_MAP = {

        "safety":
            "Strong safety sentiment from recent reviews",

        "location":
            "Excellent location convenience for this traveler profile",

        "business":
            "Highly rated by business travelers",

        "family":
            "Consistently praised by families",

        "wellness":
            "Excellent wellness and relaxation facilities",

        "service":
            "Exceptional service quality",

        "culture":
            "Authentic local experiences nearby",

        "value":
            "Strong value-for-money proposition",

        "cleanliness":
            "Consistently high cleanliness standards",

        "nightlife":
            "Excellent nightlife access",

        "beach":
            "Excellent beach experience",

        "accessibility":
            "Good accessibility support"
    }

    profile = (
        profile_vectors[
            profile_vectors["profile_id"] == profile_id
        ]
        .iloc[0]
    )

    top_aspects = (
        profile[ASPECTS]
        .sort_values(
            ascending=False
        )
        .head(3)
        .index
        .tolist()
    )

    strengths = []
    reasons = []

    for aspect in top_aspects:

        score = hotel_matrix.loc[
            hotel_id,
            aspect
        ]

        strengths.append({
            "aspect": aspect,
            "score": round(
                float(score),
                2
            )
        })

        reasons.append(
            REASON_MAP.get(
                aspect,
                f"Strong {aspect} performance"
            )
        )

    contradictions = (
        hotel_contradictions[
            hotel_contradictions["hotel_id"]
            == hotel_id
        ]
    )

    warnings = []

    for _, row in contradictions.iterrows():

        warnings.append(
            f"Mixed reviews for {row['aspect']} among {row['worst_for']} travelers"
        )

    # --------------------------------------------------
    # Confidence Score Normalization
    # --------------------------------------------------

    raw_confidence = float(
        hotel_confidence[
            hotel_confidence["hotel_id"]
            == hotel_id
        ]["confidence_score"]
        .iloc[0]
    )

    confidence_score = (
        (
            raw_confidence
            -
            hotel_confidence[
                "confidence_score"
            ].min()
        )
        /
        (
            hotel_confidence[
                "confidence_score"
            ].max()
            -
            hotel_confidence[
                "confidence_score"
            ].min()
        )
    ) * 100

    # --------------------------------------------------
    # Trend Interpretation
    # --------------------------------------------------

    hotel_trend_subset = (
        hotel_trends[
            hotel_trends["hotel_id"]
            == hotel_id
        ]
    )

    if len(hotel_trend_subset):

        avg_trend = (
            hotel_trend_subset[
                "trend_score"
            ]
            .mean()
        )

        if avg_trend > 0.10:
            trend = "improving"

        elif avg_trend < -0.10:
            trend = "declining"

        else:
            trend = "stable"

    else:
        trend = "stable"

    return (
        strengths,
        warnings,
        reasons,
        round(confidence_score, 2),
        trend
    )

CELL 12 — Final Recommendation Objects

In [ ]:
recommendations = []

for profile_id in profiles["profile_id"]:

    ranked = rank_hotels(
        profile_id
    )

    for rank, (
        hotel_id,
        score
    ) in enumerate(
        ranked,
        start=1
    ):

        (
            strengths,
            warnings,
            reasons,
            confidence_score,
            trend
        ) = generate_explanation(
            profile_id,
            hotel_id
        )

        recommendations.append({

            "profile_id":
                profile_id,

            "rank":
                rank,

            "hotel_id":
                hotel_id,

            "relevance_score":
                round(
                    float(score),
                    2
                ),

            "match_reasons":
                reasons,

            "top_strengths":
                strengths,

            "warnings":
                warnings,

            "confidence_score":
                round(
                    confidence_score,
                    2
                ),

            "trend":
                trend
        })

recommendations_df = pd.DataFrame(
    recommendations
)

recommendations_df.head()

,profile_id,rank,hotel_id,relevance_score,match_reasons,top_strengths,warnings,confidence_score,trend
0,P01,1,H017,100.00,[Excellent location convenience for this trave...,"[{'aspect': 'location', 'score': 3.49}, {'aspe...",[Mixed reviews for family among business trave...,10.17,stable
1,P01,2,H116,92.49,[Excellent location convenience for this trave...,"[{'aspect': 'location', 'score': 3.18}, {'aspe...",[],90.88,stable
2,P01,3,H048,90.57,[Excellent location convenience for this trave...,"[{'aspect': 'location', 'score': 3.47}, {'aspe...",[Mixed reviews for accessibility among family ...,30.30,stable
3,P01,4,H035,90.43,[Excellent location convenience for this trave...,"[{'aspect': 'location', 'score': 3.69}, {'aspe...",[],13.46,improving
4,P01,5,H041,89.06,[Excellent location convenience for this trave...,"[{'aspect': 'location', 'score': 2.83}, {'aspe...",[],61.77,stable


CELL 13 — Save Artifacts

In [ ]:
profile_vectors.to_parquet(
    f"{ARTIFACT_DIR}/profile_vectors.parquet",
    index=False
)

recommendations_df.to_parquet(
    f"{ARTIFACT_DIR}/recommendations.parquet",
    index=False
)

with open(
    f"{MODEL_DIR}/recommendation_engine.pkl",
    "wb"
) as f:

    pickle.dump(
        rank_hotels,
        f
    )

print(
    "Recommendation Engine Saved"
)

Recommendation Engine Saved


In [ ]:
profile_vectors.head()


,profile_id,safety,location,culture,business,family,wellness,service,cleanliness,value,accessibility,nightlife,beach
0,P01,0.617601,0.774691,0.498354,0.044221,0.094415,0.251057,0.359167,0.376931,0.250539,0.333504,0.401347,0.283661
1,P02,0.917821,0.371810,0.060393,0.907844,0.248154,0.220870,0.454044,0.439942,0.526799,0.313545,0.249454,0.328233
2,P03,0.118130,0.465055,0.189511,0.035635,0.809799,0.063618,0.293109,0.280459,0.255046,0.259976,0.229217,0.286888
3,P04,0.369761,0.326262,0.321475,0.239117,0.210807,0.814477,0.691761,0.511818,0.367636,0.127091,0.414505,0.211001
4,P05,0.633918,0.676346,0.596043,0.086687,0.159204,0.305519,0.414770,0.177883,1.000000,0.141858,0.544150,0.299314


In [ ]:
recommendations_df.head()

,profile_id,rank,hotel_id,relevance_score,match_reasons,top_strengths,warnings,confidence_score,trend
0,P01,1,H017,100.00,"[Strong location performance, Strong safety pe...","[{'aspect': 'location', 'score': 3.49}, {'aspe...",[Mixed reviews for family among business trave...,5.21,stable
1,P01,2,H035,91.03,"[Strong location performance, Strong safety pe...","[{'aspect': 'location', 'score': 3.69}, {'aspe...",[],5.30,improving
2,P01,3,H048,89.20,"[Strong location performance, Strong safety pe...","[{'aspect': 'location', 'score': 3.47}, {'aspe...",[Mixed reviews for accessibility among family ...,5.76,stable
3,P01,4,H116,88.63,"[Strong location performance, Strong safety pe...","[{'aspect': 'location', 'score': 3.18}, {'aspe...",[],7.38,stable
4,P01,5,H041,86.25,"[Strong location performance, Strong safety pe...","[{'aspect': 'location', 'score': 2.83}, {'aspe...",[],6.60,stable


In [ ]:
recommendations_df.iloc[0]

,0
profile_id,P01
rank,1
hotel_id,H017
relevance_score,100.0
match_reasons,"[Strong location performance, Strong safety pe..."
top_strengths,"[{'aspect': 'location', 'score': 3.49}, {'aspe..."
warnings,[Mixed reviews for family among business trave...
confidence_score,5.21
trend,stable


In [ ]:
profile_id = "P01"
hotel_id = "H017"

profile = profile_vectors[
    profile_vectors["profile_id"] == profile_id
].iloc[0]

print("Top profile priorities:")
print(
    profile[ASPECTS]
    .sort_values(ascending=False)
    .head(5)
)

print("\nHotel scores:")
print(
    hotel_matrix.loc[hotel_id]
    .sort_values(ascending=False)
    .head(5)
)

Top profile priorities:
location       0.774691
safety         0.617601
culture        0.498354
nightlife      0.401347
cleanliness    0.376931
Name: 0, dtype: object

Hotel scores:
aspect
nightlife    4.345397
culture      3.691489
safety       3.584609
beach        3.565983
location     3.491578
Name: H017, dtype: float64


In [ ]:
print(
    hotel_matrix.loc["H116"]
    .sort_values(ascending=False)
    .head(5)
)

aspect
nightlife    4.220654
wellness     4.064057
business     3.624452
safety       3.464059
culture      3.421103
Name: H116, dtype: float64


In [ ]:
print(profiles.iloc[0]["description"])

Solo female traveler, safety-conscious, wants a central, walkable base, keen on local culture and markets.
